# Install Prophet Library

In [ ]:
!pip install prophet

# Load Data

In [ ]:
import pandas as pd

df = pd.read_csv("propertyfinder_final.csv")

/tmp/ipykernel_1168/851466460.py:3: DtypeWarning: Columns (18,19,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("propertyfinder_final.csv")


# Convert Listed Date Column to Datetime

In [ ]:
df["listed_date"] = pd.to_datetime(
    df["listed_date"],
    errors="coerce"
)

# Aggregate Data Monthly

In [ ]:
monthly = (
    df.groupby(
        pd.Grouper(
            key="listed_date",
            freq="M"
        )
    )["listing_id"]
    .nunique()
    .reset_index()
)

monthly.columns = ["ds","y"]

/tmp/ipykernel_1168/3568783429.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  pd.Grouper(


# Display Last 10 Rows of Monthly Data

In [ ]:
monthly.tail(10)

,ds,y
36,2025-06-30 00:00:00+00:00,20
37,2025-07-31 00:00:00+00:00,35
38,2025-08-31 00:00:00+00:00,30
39,2025-09-30 00:00:00+00:00,46
40,2025-10-31 00:00:00+00:00,92
41,2025-11-30 00:00:00+00:00,119
42,2025-12-31 00:00:00+00:00,207
43,2026-01-31 00:00:00+00:00,523
44,2026-02-28 00:00:00+00:00,18566
45,2026-03-31 00:00:00+00:00,16318


# Filter Data for 2026

In [ ]:
monthly_2026 = monthly[
    monthly["ds"] >= "2026-02-01"
].copy()

# Display 2026 Data

In [ ]:
monthly_2026

,ds,y
44,2026-02-28 00:00:00+00:00,18566
45,2026-03-31 00:00:00+00:00,16318


# Remove Timezone Information

In [ ]:
monthly_2026["ds"] = pd.to_datetime(
    monthly_2026["ds"]
).dt.tz_localize(None)

# Initialize and Train Prophet Model

In [ ]:
from prophet import Prophet

model = Prophet(
    growth='linear',
    yearly_seasonality=False,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.5
)

model.fit(monthly_2026)

INFO:prophet:n_changepoints greater than number of observations. Using 0.


# Create Future Dataframe and Predict

In [ ]:
future = model.make_future_dataframe(
    periods=24,
    freq='M'
)

forecast = model.predict(future)

/usr/local/lib/python3.12/dist-packages/prophet/forecaster.py:1875: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(


# Clip Forecast Values to Prevent Negatives

In [ ]:
forecast["yhat"] = forecast["yhat"].clip(lower=0)

forecast["yhat_lower"] = forecast["yhat_lower"].clip(lower=0)

forecast["yhat_upper"] = forecast["yhat_upper"].clip(lower=0)

# Prepare and Export Forecast Results

In [ ]:
forecast_result = forecast[
    [
        "ds",
        "yhat",
        "yhat_lower",
        "yhat_upper"
    ]
]

forecast_result.to_csv(
    "Forecast_2026_2028.csv",
    index=False
)

# Download Forecast CSV

In [ ]:
from google.colab import files

files.download(
    "Forecast_2026_2028.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Calculate Optimistic, Realistic, and Pessimistic Scenarios

In [ ]:
base = 16332

optimistic = []
realistic = []
pessimistic = []

for i in range(24):
    optimistic.append(round(base * (1.08 ** (i+1))))
    realistic.append(round(base * (1.05 ** (i+1))))
    pessimistic.append(round(base * (1.02 ** (i+1))))

# Generate Future Dates for Business Scenario

In [ ]:
future_dates = pd.date_range(
    start='2026-04-30',
    periods=24,
    freq='M'
)

/tmp/ipykernel_32909/2659752891.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  future_dates = pd.date_range(


# Create Business Forecast Dataframe

In [ ]:
forecast_df = pd.DataFrame({
    'ds': future_dates,
    'Optimistic': optimistic,
    'Realistic': realistic,
    'Pessimistic': pessimistic
})

# Download Business Forecast CSV

In [ ]:
from google.colab import files

files.download(
    'Business_Forecast.csv'
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>